# ENGG2112: Random Forest model

Train and evaluate a Random Forest classifier for the APTOS 0–4 task using pooled features from a frozen ImageNet ResNet-18. This is a **single-model notebook**; it does not train the CNN-BiLSTM or the fine-tuned ResNet-18 baseline. It reuses the project’s prepared data and fixed group-aware split. Outputs are validation results; the test set stays locked until review and split freeze.

## Setup and configuration

Use the prepared APTOS handoff and existing group-aware split. This notebook does not download data or create a new split. Set `DATA_ROOT` to the prepared handoff root if it is not at the default path. PyTorch, TorchVision, and scikit-learn must be installed in the selected notebook kernel. The first run may download official ImageNet ResNet-18 weights; do not disable certificate checks.


In [1]:
from pathlib import Path
import hashlib
import json
import random

import numpy as np
import pandas as pd
from PIL import Image

try:
    import torch
    from torch import nn
    from torch.utils.data import DataLoader
    from torchvision import models, transforms
    from torchvision.models import ResNet18_Weights
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.metrics import (accuracy_score, balanced_accuracy_score, classification_report,
                                 confusion_matrix, cohen_kappa_score, f1_score)
except ImportError as exc:
    raise ImportError(
        'Install PyTorch, TorchVision, NumPy, pandas, Pillow, and scikit-learn '
        'in the selected notebook kernel, then restart the kernel.'
    ) from exc

SEED = 2112
REPEAT_SEEDS = [2112, 2113, 2114]
DATA_ROOT = Path.cwd() / 'artifacts/notebook_runs/notebook_run_001'
OUTPUT_DIR = Path.cwd() / 'outputs' / 'aptos_random_forest'
FEATURE_CACHE = Path.cwd() / 'outputs' / 'aptos_shared_features' / 'resnet18_grid_features.npz'
BATCH_SIZE = 32
RF_N_ESTIMATORS = 500
RF_MIN_SAMPLES_LEAF = 2
CONFIRM_REVIEWED_FROZEN_SPLIT = False
TEST_EVALUATION_COMPLETED = False

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('mps' if torch.backends.mps.is_available()
                      else 'cuda' if torch.cuda.is_available() else 'cpu')

SPLIT_PATH = DATA_ROOT / 'artifacts/splits/split_manifest.csv'
PREPARED_DIR = DATA_ROOT / 'data/processed_224_v1'
PREPARED_MANIFEST_PATH = PREPARED_DIR / 'processed_manifest.csv'
PREPROCESSING_CONFIG_PATH = PREPARED_DIR / 'preprocessing_config.json'
missing = [str(path) for path in (SPLIT_PATH, PREPARED_MANIFEST_PATH, PREPROCESSING_CONFIG_PATH)
           if not path.is_file()]
if missing:
    raise FileNotFoundError('Prepared APTOS handoff is incomplete. Set DATA_ROOT to the handoff root. Missing: '
                            + ', '.join(missing))

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FEATURE_CACHE.parent.mkdir(parents=True, exist_ok=True)
def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

MANIFEST_SHA256 = sha256_file(SPLIT_PATH)
print('Device:', DEVICE, '| Torch:', torch.__version__, '| TorchVision:', __import__('torchvision').__version__)
print('Handoff:', DATA_ROOT, '| Split manifest SHA-256:', MANIFEST_SHA256)

Device: mps | Torch: 2.14.1 | TorchVision: 0.29.1
Handoff: /Users/quangduyvu1723/Documents/GitHub/ENGG2112-final-project/artifacts/notebook_runs/notebook_run_001 | Split manifest SHA-256: 81283b51d8274a7d4ee5cd873cfc6479fd9efd178a75bacd45e297dd41d57a54


## Shared frozen image features

The frozen ImageNet-pretrained ResNet-18 encoder creates 49 spatial cells with 512 channels for each 224 × 224 image. The Random Forest receives the mean across those cells (512 features). No labels are used to fit the encoder. The feature cache is local, ignored by Git, and checked against the split-manifest hash and encoder name.


In [2]:
split = pd.read_csv(SPLIT_PATH, dtype={'id_code': str, 'group_id': str, 'pixel_sha256': str})
prepared = pd.read_csv(PREPARED_MANIFEST_PATH, dtype={'id_code': str})
split_columns = {'id_code', 'diagnosis', 'split', 'group_id', 'pixel_sha256'}
prepared_columns = {'id_code', 'diagnosis', 'split', 'relative_path'}
if not split_columns <= set(split.columns) or not prepared_columns <= set(prepared.columns):
    raise ValueError('Handoff manifests do not satisfy the expected columns.')
for label, frame in [('split', split), ('prepared', prepared)]:
    if frame.id_code.isna().any() or frame.id_code.duplicated().any():
        raise ValueError(f'{label} manifest contains missing or duplicate IDs.')
    if not pd.to_numeric(frame.diagnosis, errors='coerce').isin(range(5)).all():
        raise ValueError(f'{label} manifest contains an invalid grade.')
    if not frame.split.isin(['train', 'validation', 'test']).all():
        raise ValueError(f'{label} manifest contains an invalid split.')
if set(split.id_code) != set(prepared.id_code):
    raise ValueError('Split and prepared manifests have different image IDs.')
joined = split.merge(prepared, on='id_code', validate='one_to_one', suffixes=('', '_prepared'))
if not joined.diagnosis.eq(joined.diagnosis_prepared).all() or not joined.split.eq(joined.split_prepared).all():
    raise ValueError('Split and prepared manifests disagree on labels or assignments.')
for column in ('group_id', 'pixel_sha256'):
    if split.groupby(column).split.nunique().max() != 1:
        raise ValueError(f'{column} crosses split boundaries.')
if not joined.relative_path.eq(joined.id_code + '.png').all():
    raise ValueError('Prepared image paths must be ID.png.')
preprocessing = json.loads(PREPROCESSING_CONFIG_PATH.read_text(encoding='utf-8'))
if preprocessing.get('size') != 224 or preprocessing.get('mode') != 'RGB':
    raise ValueError('Expected prepared 224 × 224 RGB images.')
joined['diagnosis'] = joined.diagnosis.astype(int)
joined['image_path'] = [str(PREPARED_DIR / name) for name in joined.relative_path]
missing_images = [path for path in joined.image_path if not Path(path).is_file()]
if missing_images:
    raise FileNotFoundError(f'{len(missing_images)} prepared images are missing.')
joined = joined.sort_values('id_code').reset_index(drop=True)
train_frame = joined.loc[joined.split.eq('train')].reset_index(drop=True)
validation_frame = joined.loc[joined.split.eq('validation')].reset_index(drop=True)
test_frame = joined.loc[joined.split.eq('test')].reset_index(drop=True)
if min(len(train_frame), len(validation_frame), len(test_frame)) == 0:
    raise ValueError('Train, validation, and test splits must all be non-empty.')
print('Rows:', len(joined), '| train:', len(train_frame), '| validation:', len(validation_frame),
      '| test locked:', len(test_frame))
display(pd.crosstab(joined.split, joined.diagnosis).reindex(columns=range(5), fill_value=0))

ENCODER_ID = ResNet18_Weights.DEFAULT.name
feature_frame = joined.loc[joined.split.isin(['train', 'validation'])].reset_index(drop=True)

class ImageDataset(torch.utils.data.Dataset):
    def __init__(self, frame):
        self.frame = frame
        self.transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
        ])
    def __len__(self):
        return len(self.frame)
    def __getitem__(self, index):
        row = self.frame.iloc[index]
        with Image.open(row.image_path) as image:
            image = image.convert('RGB')
            if image.size != (224, 224):
                raise ValueError(f'Unexpected image size for {row.id_code}: {image.size}')
            return self.transform(image)

_encoder = None
def extract_feature_sequences(frame):
    global _encoder
    if _encoder is None:
        encoder_model = models.resnet18(weights=ResNet18_Weights.DEFAULT)
        _encoder = nn.Sequential(*list(encoder_model.children())[:-2]).to(DEVICE).eval()
        for parameter in _encoder.parameters():
            parameter.requires_grad_(False)
    image_loader = DataLoader(ImageDataset(frame), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    batches = []
    with torch.inference_mode():
        for batch_number, images in enumerate(image_loader, start=1):
            fmap = _encoder(images.to(DEVICE))
            batches.append(fmap.flatten(2).transpose(1, 2).cpu().numpy().astype(np.float32))
            if batch_number % 20 == 0 or batch_number == len(image_loader):
                print(f'Encoded {min(batch_number * BATCH_SIZE, len(frame))}/{len(frame)} permitted images')
    return np.concatenate(batches, axis=0)

cache_valid = False
if FEATURE_CACHE.is_file():
    with np.load(FEATURE_CACHE, allow_pickle=False) as cached:
        cache_metadata = json.loads(str(cached['metadata'].item()))
        cache_valid = (cache_metadata.get('manifest_sha256') == MANIFEST_SHA256
                       and cache_metadata.get('encoder') == ENCODER_ID
                       and cache_metadata.get('included_splits') == ['train', 'validation']
                       and np.array_equal(cached['id_code'].astype(str), feature_frame.id_code.to_numpy(dtype=str)))

if cache_valid:
    with np.load(FEATURE_CACHE, allow_pickle=False) as cached:
        feature_sequences = cached['features'].astype(np.float32)
        feature_labels = cached['labels'].astype(np.int64)
        feature_ids = cached['id_code'].astype(str)
    print('Reused validated train/validation feature cache:', FEATURE_CACHE)
else:
    feature_sequences = extract_feature_sequences(feature_frame)
    feature_labels = feature_frame.diagnosis.to_numpy(dtype=np.int64)
    feature_ids = feature_frame.id_code.to_numpy(dtype=str)
    metadata = {'manifest_sha256': MANIFEST_SHA256, 'encoder': ENCODER_ID,
                'feature_shape_per_image': list(feature_sequences.shape[1:]),
                'included_splits': ['train', 'validation'],
                'normalization': 'ImageNet mean/std', 'augmentation': 'none'}
    np.savez_compressed(FEATURE_CACHE, features=feature_sequences, labels=feature_labels,
                        id_code=feature_ids, metadata=json.dumps(metadata))
    print('Saved local train/validation feature cache:', FEATURE_CACHE)

if feature_sequences.shape != (len(feature_frame), 49, 512):
    raise ValueError(f'Expected (N, 49, 512) CNN feature sequences; got {feature_sequences.shape}.')
if not np.array_equal(feature_ids, feature_frame.id_code.to_numpy(dtype=str)):
    raise ValueError('Feature cache IDs are not in current manifest order.')
split_indices = {name: np.flatnonzero(feature_frame.split.to_numpy() == name)
                  for name in ('train', 'validation')}
X_sequence = {name: feature_sequences[idx] for name, idx in split_indices.items()}
y = {name: feature_labels[idx] for name, idx in split_indices.items()}
X_pooled = {name: values.mean(axis=1) for name, values in X_sequence.items()}
print('Test images have not been opened or encoded.')
print('Pooled RF feature shape:', X_pooled['train'].shape,
      '| CNN-BiLSTM sequence shape:', X_sequence['train'].shape)

Rows: 3600 | train: 2521 | validation: 538 | test locked: 541


diagnosis,0,1,2,3,4
split,,,,,
test,273,53,144,27,44
train,1258,253,684,127,199
validation,270,54,148,26,40


Reused validated train/validation feature cache: /Users/quangduyvu1723/Documents/GitHub/ENGG2112-final-project/outputs/aptos_shared_features/resnet18_grid_features.npz
Test images have not been opened or encoded.
Pooled RF feature shape: (2521, 512) | CNN-BiLSTM sequence shape: (2521, 49, 512)


## Random Forest method

Random Forest is a classical ensemble-learning classifier, not a neural network. It fits 500 decision trees to pooled features from the frozen pretrained CNN. `class_weight='balanced_subsample'` addresses the training-grade imbalance. The configuration is fixed in advance; three random seeds quantify some estimator variability. The primary result is seed 2112, selected before evaluation and not selected by validation/test score.


In [3]:
CLASS_IDS = list(range(5))
CLASS_NAMES = ['No DR', 'Mild', 'Moderate', 'Severe', 'Proliferative DR']

def calculate_metrics(y_true, y_pred):
    return {
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'balanced_accuracy': float(balanced_accuracy_score(y_true, y_pred)),
        'macro_f1': float(f1_score(y_true, y_pred, labels=CLASS_IDS, average='macro', zero_division=0)),
        'quadratic_weighted_kappa': float(cohen_kappa_score(y_true, y_pred, labels=CLASS_IDS, weights='quadratic')),
        'per_class': classification_report(y_true, y_pred, labels=CLASS_IDS, target_names=CLASS_NAMES,
                                           output_dict=True, zero_division=0),
        'confusion_matrix': confusion_matrix(y_true, y_pred, labels=CLASS_IDS).tolist(),
        'confusion_matrix_normalized_true': np.nan_to_num(
            confusion_matrix(y_true, y_pred, labels=CLASS_IDS, normalize='true')).tolist(),
    }

rf_results = []
rf_models = {}
for seed in REPEAT_SEEDS:
    forest = RandomForestClassifier(
        n_estimators=RF_N_ESTIMATORS,
        min_samples_leaf=RF_MIN_SAMPLES_LEAF,
        max_features='sqrt',
        class_weight='balanced_subsample',
        random_state=seed,
        n_jobs=-1,
    )
    forest.fit(X_pooled['train'], y['train'])
    prediction = forest.predict(X_pooled['validation'])
    metrics = calculate_metrics(y['validation'], prediction)
    rf_models[seed] = forest
    rf_results.append({'method': 'Random Forest', 'seed': seed,
                       **{key: metrics[key] for key in ('accuracy', 'balanced_accuracy',
                                                        'macro_f1', 'quadratic_weighted_kappa')},
                       'weighted_f1': metrics['per_class']['weighted avg']['f1-score']})
    if seed == REPEAT_SEEDS[0]:
        rf_primary_validation = metrics
    print(f'RF seed {seed}: accuracy={metrics["accuracy"]:.4f}, '
          f'balanced_accuracy={metrics["balanced_accuracy"]:.4f}, '
          f'macro_f1={metrics["macro_f1"]:.4f}, QWK={metrics["quadratic_weighted_kappa"]:.4f}')

rf_results_frame = pd.DataFrame(rf_results)
display(rf_results_frame)
display(pd.DataFrame(rf_primary_validation['per_class']).T)
display(pd.DataFrame(rf_primary_validation['confusion_matrix'], index=CLASS_NAMES, columns=CLASS_NAMES))

rf_metric_names = ['accuracy', 'balanced_accuracy', 'macro_f1', 'weighted_f1', 'quadratic_weighted_kappa']
rf_summary = {
    metric: {'mean': float(rf_results_frame[metric].mean()),
             'std': float(rf_results_frame[metric].std(ddof=1))}
    for metric in rf_metric_names
}
rf_payload = {
    'model': 'Random Forest on pooled frozen ResNet-18 features',
    'split_manifest_sha256': MANIFEST_SHA256,
    'encoder': ENCODER_ID,
    'repeat_seeds': REPEAT_SEEDS,
    'primary_seed': REPEAT_SEEDS[0],
    'settings': {'n_estimators': RF_N_ESTIMATORS, 'min_samples_leaf': RF_MIN_SAMPLES_LEAF,
                 'max_features': 'sqrt', 'class_weight': 'balanced_subsample'},
    'validation_runs': rf_results,
    'validation_summary': rf_summary,
    'primary_validation': rf_primary_validation,
    'test_evaluation': 'not run; review and freeze the split first',
}
RF_RESULTS_PATH = OUTPUT_DIR / 'validation_metrics.json'
RF_RESULTS_PATH.write_text(json.dumps(rf_payload, indent=2), encoding='utf-8')
print('Random Forest validation results saved to:', RF_RESULTS_PATH)


RF seed 2112: accuracy=0.7658, balanced_accuracy=0.4391, macro_f1=0.4286, QWK=0.7928
RF seed 2113: accuracy=0.7714, balanced_accuracy=0.4517, macro_f1=0.4510, QWK=0.8009
RF seed 2114: accuracy=0.7677, balanced_accuracy=0.4416, macro_f1=0.4324, QWK=0.7959


,method,seed,accuracy,balanced_accuracy,macro_f1,quadratic_weighted_kappa,weighted_f1
0,Random Forest,2112,0.765799,0.439089,0.428604,0.792760,0.717734
1,Random Forest,2113,0.771375,0.451736,0.451048,0.800910,0.727323
2,Random Forest,2114,0.767658,0.441571,0.432389,0.795931,0.720732


,precision,recall,f1-score,support
No DR,0.946043,0.974074,0.959854,270.000000
Mild,0.608696,0.259259,0.363636,54.000000
Moderate,0.573276,0.898649,0.700000,148.000000
Severe,1.000000,0.038462,0.074074,26.000000
Proliferative DR,0.250000,0.025000,0.045455,40.000000
accuracy,0.765799,0.765799,0.765799,0.765799
macro avg,0.675603,0.439089,0.428604,538.000000
weighted avg,0.760495,0.765799,0.717734,538.000000


,No DR,Mild,Moderate,Severe,Proliferative DR
No DR,263,1,6,0,0
Mild,6,14,34,0,0
Moderate,8,7,133,0,0
Severe,1,0,21,1,3
Proliferative DR,0,1,38,0,1


Random Forest validation results saved to: /Users/quangduyvu1723/Documents/GitHub/ENGG2112-final-project/outputs/aptos_random_forest/validation_metrics.json


## Optional final test evaluation — locked by default

Only after the team reviews near-duplicate candidates, documents decisions, and freezes the split, set `CONFIRM_REVIEWED_FROZEN_SPLIT = True` in the configuration cell and run this cell once. Seed 2112 is the predetermined RF result. Do not select settings or seeds using test results.


In [4]:
if not CONFIRM_REVIEWED_FROZEN_SPLIT:
    print('Test evaluation skipped: review and freeze the split before enabling it.')
elif TEST_EVALUATION_COMPLETED:
    raise RuntimeError('Test evaluation was already run in this kernel.')
else:
    TEST_EVALUATION_COMPLETED = True
    test_features = extract_feature_sequences(test_frame).mean(axis=1)
    prediction = rf_models[REPEAT_SEEDS[0]].predict(test_features)
    test_metrics = calculate_metrics(test_frame.diagnosis.to_numpy(dtype=np.int64), prediction)
    test_payload = {
        'model': 'Random Forest on pooled frozen ResNet-18 features',
        'split_manifest_sha256': MANIFEST_SHA256,
        'confirmation': 'reviewed and frozen by user before evaluation',
        'seed': REPEAT_SEEDS[0],
        'test': test_metrics,
    }
    test_path = OUTPUT_DIR / 'test_metrics_once.json'
    test_path.write_text(json.dumps(test_payload, indent=2), encoding='utf-8')
    print('Final test metrics saved to:', test_path)
    print({key: value for key, value in test_metrics.items() if key != 'per_class'})
    display(pd.DataFrame(test_metrics['per_class']).T)


Test evaluation skipped: review and freeze the split before enabling it.


## Summary and limitations

This notebook trains only the Random Forest classifier, with a frozen ResNet-18 used to extract image features. It uses the existing split and selects no model using test data. Report validation mean and standard deviation across three seeds, alongside per-grade recall/F1 and confusion matrix. This is a limited seed-sensitivity check on one provisional split, not a generalization guarantee. The project audit reported unresolved near-duplicate candidates and did not verify patient-level independence. The RF result is not directly equivalent to the end-to-end fine-tuned ResNet-18 baseline.
